## Treasury Bulletin PDF OCR Removal

Converts Treasury Bulletin PDFs from their original format (with embedded/extractable text) into no-OCR PDFs where each page is rendered as an image with the OCR removed. 

**Input:** `treasury_bulletin_pdfs/` — original Treasury Bulletin PDFs

**Output:** `treasury_bulletin_pdfs_no_ocr/` — image-only versions with no extractable text

In [ ]:
%pip install pymupdf Pillow

In [ ]:
import fitz
from PIL import Image
import io
import os
import time

SOURCE_DIR = "treasury_bulletin_pdfs"
TARGET_DIR = "treasury_bulletin_pdfs_no_ocr"
MAX_SIZE_MB = 50

In [ ]:
def convert_pdf_to_image_only(source_path, target_path, dpi=150, quality=85):
    doc = fitz.open(source_path)
    output_doc = fitz.open()

    for page_num in range(len(doc)):
        page = doc.load_page(page_num)
        pix = page.get_pixmap(dpi=dpi)

        img = Image.frombytes("RGB", [pix.width, pix.height], pix.samples)
        img_bytes = io.BytesIO()
        img.save(img_bytes, format="JPEG", quality=quality, optimize=True)
        img_bytes.seek(0)

        img_pdf = fitz.open("jpeg", img_bytes.read())
        output_doc.insert_pdf(img_pdf)

    output_doc.save(target_path)
    output_doc.close()
    doc.close()


def convert_pdf_adaptive(source_path, target_path, max_size_mb=50):
    """Convert with progressively lower quality until the output is under max_size_mb."""
    configs = [(150, 85), (120, 75), (100, 70), (80, 65), (60, 60)]

    for dpi, quality in configs:
        convert_pdf_to_image_only(source_path, target_path, dpi=dpi, quality=quality)
        size_mb = os.path.getsize(target_path) / (1024 * 1024)
        if size_mb <= max_size_mb:
            return size_mb, dpi, quality

    return size_mb, dpi, quality

In [ ]:
os.makedirs(TARGET_DIR, exist_ok=True)

pdf_files = sorted([f for f in os.listdir(SOURCE_DIR) if f.lower().endswith(".pdf")])
print(f"Found {len(pdf_files)} PDF files")

start_time = time.time()

for i, pdf_file in enumerate(pdf_files, 1):
    source_path = os.path.join(SOURCE_DIR, pdf_file)
    target_path = os.path.join(TARGET_DIR, pdf_file)

    try:
        size_mb, dpi, quality = convert_pdf_adaptive(source_path, target_path, max_size_mb=MAX_SIZE_MB)
        print(f"[{i:3d}/{len(pdf_files)}] {pdf_file} -> {size_mb:.1f} MB (dpi={dpi}, q={quality})")
    except Exception as e:
        print(f"[{i:3d}/{len(pdf_files)}] {pdf_file} -> ERROR: {e}")

    if i % 50 == 0:
        elapsed = time.time() - start_time
        remaining = (len(pdf_files) - i) * (elapsed / i) / 60
        print(f"    Progress: {i}/{len(pdf_files)} — ETA: {remaining:.1f} min")

total_min = (time.time() - start_time) / 60
print(f"\nDone. {len(pdf_files)} files in {total_min:.1f} minutes.")

In [ ]:
import random

converted = [f for f in os.listdir(TARGET_DIR) if f.lower().endswith(".pdf")]
print(f"Source: {len(pdf_files)} files")
print(f"Target: {len(converted)} files")

sample = random.sample(converted, min(5, len(converted)))
for fname in sample:
    orig_path = os.path.join(SOURCE_DIR, fname)
    conv_path = os.path.join(TARGET_DIR, fname)

    orig_doc = fitz.open(orig_path)
    orig_text = orig_doc.load_page(0).get_text()
    orig_doc.close()

    conv_doc = fitz.open(conv_path)
    conv_text = conv_doc.load_page(0).get_text()
    conv_doc.close()

    conv_size = os.path.getsize(conv_path) / (1024 * 1024)
    status = "OK" if len(conv_text) == 0 else f"WARN: {len(conv_text)} chars remain"
    print(f"  {fname}: original={len(orig_text)} chars, converted={len(conv_text)} chars, size={conv_size:.1f} MB — {status}")